In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
import copy
import time
import math
import statistics
import random
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from torch_geometric.nn import GCNConv
from scipy.sparse import coo_matrix
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import evaluate # For BLEU calculation

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# NOTE: Replace these with your actual paths
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_objects_reduced.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

In [4]:
# --- Metadata Dimensions (Based on your original code) ---
NUM_COLORS = 77
NUM_CATEGORIES = 53
NUM_OBJECTS = 61
D_MODEL = 256 # Transformer hidden dimension

In [5]:
# --- Granger Causality Matrix Creation ---
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j: continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            # Ensure data has enough non-NaN values for grangercausalitytests
            data = np.vstack([ts_j, ts_i]).T
            if len(data) < 10: continue
                
            try:
                # maxlag=5 is used for the test
                results = grangercausalitytests(data, maxlag=5, verbose=False)
                # We use the F-test result from the 5th lag
                p_value = results[5][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)
    
    return edge_index.to(torch.long), edge_attr.to(torch.float)

In [6]:
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype('float32'))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))
        
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# --- Transformer Core Components (Utility) ---
def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linears = get_clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        batch_size = query.size(0)
        
        query, key, value = [l(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
                             for l, x in zip(self.linears, (query, key, value))]

        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        p_attn = F.softmax(scores, dim=-1)
        p_attn = self.dropout(p_attn)
        
        x = torch.matmul(p_attn, value)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.linears[-1](x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, d_meta):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Metadata Conditional Bias (Semantic Gating)
        self.meta_gate = nn.Sequential(
            nn.Linear(d_meta, d_model),
            nn.Sigmoid()
        )
        
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask, meta_features):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, memory, memory, src_mask)))
        
        # Apply Metadata Conditional Bias (Semantic Gating)
        # meta_features: (B, D_meta)
        gate = self.meta_gate(meta_features).unsqueeze(1) # (B, 1, D_model)
        x = x * gate 
        
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x

# --- Metadata Encoder ---
class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_categories, num_objects, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.category_embedding = nn.Embedding(num_categories, category_emb_dim)
        
        # NOTE: This processor expects the raw 61-dim multi-hot vector for objects
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )
        
        self.output_dim = color_emb_dim + category_emb_dim + object_feature_dim
        self.color_emb_dim = color_emb_dim
        self.category_emb_dim = category_emb_dim
        self.object_feature_dim = object_feature_dim

    def forward(self, metadata):
        # metadata is (B, 2 + NUM_OBJECTS)
        color_ids = metadata[:, 0].long()
        category_ids = metadata[:, 1].long()
        object_features_raw = metadata[:, 2:]
        
        object_features_raw = object_features_raw.float()

        color_vec = self.color_embedding(color_ids)
        category_vec = self.category_embedding(category_ids)
        object_vec = self.object_processor(object_features_raw) # (B, 128)

        combined_features = torch.cat([color_vec, category_vec, object_vec], dim=1)
        return combined_features

# --- SpatioTemporal EEG Encoder (GCN + Transformer Encoder) ---
class SpatioTemporalEEGEncoderTF(nn.Module):
    def __init__(self, num_channels=62, d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, dropout=0.1):
        super().__init__()
        self.num_channels = num_channels
        self.d_model = d_model
        
        # GCN layers convert C channels into D_MODEL features
        self.gcn1 = GCNConv(num_channels, d_model)
        self.gcn2 = GCNConv(d_model, d_model)
        self.spatial_dropout = nn.Dropout(dropout)

        self.pos_encoding = PositionalEncoding(d_model)

        encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
        self.transformer_layers = get_clones(encoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, eeg, edge_index, edge_attr):
        batch_size, num_channels, num_timesteps = eeg.shape
        
        # --- Spatial Processing (GCN) ---
        batch_edge_index, batch_edge_attr = self._prepare_gcn_input(batch_size, num_timesteps, edge_index, edge_attr)
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels) # (B*T, C)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.spatial_dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))
        
        x = x.reshape(batch_size, num_timesteps, self.d_model)
        
        # --- Transformer Input Preparation ---
        cls_token = self.cls_token.repeat(batch_size, 1, 1) # (B, 1, D_model)
        x = torch.cat([cls_token, x], dim=1) # (B, T+1, D_model)

        x = self.pos_encoding(x)

        # --- Temporal Processing (Transformer Encoder Stack) ---
        for layer in self.transformer_layers:
            x = layer(x)
        
        return self.layer_norm(x)

    def _prepare_gcn_input(self, batch_size, num_timesteps, edge_index, edge_attr):        
        # This handles graph batching by stacking nodes (B*C) and duplicating edges
        batch_edge_index = edge_index.repeat(1, batch_size * num_timesteps) 
        batch_edge_attr = edge_attr.repeat(batch_size * num_timesteps)
        
        batch_offset = torch.arange(batch_size * num_timesteps, device=edge_index.device) * self.num_channels
        batch_edge_index += batch_offset.repeat_interleave(edge_index.shape[1]).unsqueeze(0)
        return batch_edge_index, batch_edge_attr # Simplified batching based on original concept

# --- Transformer Decoder ---
class DecoderTF(nn.Module):
    def __init__(self, vocab_size, emb_dim, d_model, num_layers, num_heads, d_ff, pad_id, dropout, d_meta):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.pos_encoding = PositionalEncoding(emb_dim)
        self.input_projection = nn.Linear(emb_dim, d_model) 
        
        decoder_layer = TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, d_meta)
        self.transformer_layers = get_clones(decoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, target_text_ids, memory, memory_mask, meta_features):
        tgt_embed = self.embedding(target_text_ids)
        x = self.pos_encoding(tgt_embed)
        x = self.input_projection(x)

        tgt_seq_len = target_text_ids.size(1)
        # Causal mask: look at only previous tokens
        tgt_mask = torch.triu(torch.ones(tgt_seq_len, tgt_seq_len), diagonal=1).bool().to(x.device)
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)
        
        for layer in self.transformer_layers:
            x = layer(x, memory, memory_mask, tgt_mask, meta_features)
        
        x = self.layer_norm(x)
        return self.fc_out(x)

In [7]:
class Seq2SeqTF(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_categories, num_objects, 
                 d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, pad_id=PAD_ID, dropout=0.1, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        
        self.encoder = SpatioTemporalEEGEncoderTF(
            d_model=d_model, num_layers=num_layers, num_heads=num_heads, d_ff=d_ff, dropout=dropout
        )
        self.meta_encoder = MetadataEncoder(
            num_colors, num_categories, num_objects, color_emb_dim, category_emb_dim, object_feature_dim
        )
        
        meta_features_dim = self.meta_encoder.output_dim
        
        self.decoder = DecoderTF(
            text_vocab_size, d_model, d_model, num_layers, num_heads, d_ff, pad_id, dropout, meta_features_dim
        )
        
        # Head for auxiliary prediction of metadata from the EEG CLS token
        self.meta_head = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_categories + num_objects)
        )
        self.num_colors = num_colors
        self.num_categories = num_categories
        self.num_objects = num_objects
        self.pad_id = pad_id

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr):
        # 1. EEG Encoding (UNCONDITIONAL)
        eeg_features = self.encoder(eeg, edge_index, edge_attr) # (B, T+1, D_model)
        
        # 2. Metadata Encoding (CONDITIONAL - uses ground truth metadata)
        meta_features = self.meta_encoder(metadata) # (B, D_meta)
        
        # 3. Auxiliary Metadata Prediction
        cls_token_feature = eeg_features[:, 0, :] # (B, D_model)
        meta_preds = self.meta_head(cls_token_feature)
        
        # 4. Decoding
        decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
        memory_mask = None 
        
        # The decoder is conditioned on the ground truth meta_features during training
        text_logits = self.decoder(
            target_text[:, :-1], # Target input (shifted right)
            decoder_memory, 
            memory_mask, 
            meta_features # Ground Truth Meta features
        )
        
        pred_color = meta_preds[:, :self.num_colors]
        pred_category = meta_preds[:, self.num_colors:self.num_colors + self.num_categories]
        pred_object = meta_preds[:, self.num_colors + self.num_categories:]
        
        return text_logits, pred_color, pred_category, pred_object

# --- Training and Evaluation Functions ---
def train_one_epoch_tf(model, loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training TF", leave=False)
    
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        optimizer.zero_grad()
        
        # Conditional forward pass (uses ground truth meta_b)
        text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix(text_loss=text_loss.item(), meta_loss=loss.item() - text_loss.item())
        
    return total_loss / len(loader)

def evaluate_tf(model, loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for eeg_b, meta_b, txt_b in loader:
            eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
            
            # Conditional forward pass (uses ground truth meta_b)
            text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr)
            
            text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
            color_loss = color_criterion(pred_color, meta_b[:, 0].long())
            category_loss = category_criterion(pred_category, meta_b[:, 1].long())
            object_loss = object_criterion(pred_object, meta_b[:, 2:])
            
            loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
            total_loss += loss.item()
            
    return total_loss / len(loader)

In [8]:
@torch.no_grad()
def generate_text_beam_unconditional(model, eeg_signal, edge_index, edge_attr, 
                                     beam_width=5, max_len=64):    
    """
    Generates text using Beam Search, conditioned ONLY on the EEG signal.
    It uses the model's prediction of metadata from the CLS token to create 
    the meta_features for the decoder's semantic gating.
    """
    model.eval()
    
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    
    # 1. Encode EEG (UNCONDITIONAL)
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    decoder_memory = eeg_features[:, 1:, :] 
    memory_mask = None 
    
    # 2. Predict Metadata from CLS token
    cls_token_feature = eeg_features[:, 0, :]
    meta_preds = model.meta_head(cls_token_feature) # (1, Total Meta Size)

    # 3. Create conditioning vector (M_features) from model's predictions
    
    # 3a. Predicted Color and Category IDs (argmax on logits)
    pred_color_id = meta_preds[:, :model.num_colors].argmax(dim=1)
    pred_category_id = meta_preds[:, model.num_colors:model.num_colors + model.num_categories].argmax(dim=1)
    
    # 3b. Predicted Object Logits (soft output)
    pred_object_logits = meta_preds[:, model.num_colors + model.num_categories:] # (1, NUM_OBJECTS)
    
    # 3c. Get the *learned embeddings* and *processed features* using the MetadataEncoder components
    color_vec = model.meta_encoder.color_embedding(pred_color_id)      # (1, color_emb_dim)
    category_vec = model.meta_encoder.category_embedding(pred_category_id) # (1, category_emb_dim)
    
    # Use the predicted object *logits* as input to the Object Processor, 
    # as this is the best available learned representation derived from the EEG for objects.
    object_vec = model.meta_encoder.object_processor(pred_object_logits) # (1, object_feature_dim)

    # 3d. Combine the predicted feature vectors to form the UNCONDITIONAL meta_features
    meta_features = torch.cat([color_vec, category_vec, object_vec], dim=1) # (1, D_meta)
    
    # We will also return the hard-predicted IDs for printing/comparison
    hard_pred_color_id = pred_color_id.item()
    hard_pred_category_id = pred_category_id.item()
    hard_pred_object_id = pred_object_logits.argmax().item()

    # 4. Initialization: Beams list stores (sequence_ids, cumulative_log_probability)
    initial_seq = [SOS_ID]
    beams = [(initial_seq, 0.0)]
    
    # 5. Beam Search Loop
    for _ in range(max_len):
        new_beams = []
        
        for seq, score in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score))
                continue
                
            input_ids = torch.tensor([seq], dtype=torch.long, device=device)

            # Decoder is conditioned on the UNCONDITIONAL meta_features
            logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
            next_token_logits = logits[:, -1, :].squeeze(0)
            
            log_probs = F.log_softmax(next_token_logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            for k in range(beam_width):
                new_token_id = top_ids[k].item()
                new_log_prob = top_log_probs[k].item()
                
                new_seq = seq + [new_token_id]
                new_score = score + new_log_prob 
                
                new_beams.append((new_seq, new_score))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        if beams[0][0][-1] == EOS_ID:
            break
            
    # 6. Final Output Selection
    best_seq = beams[0][0]
    
    if best_seq[-1] == EOS_ID:
        predicted_text_ids = best_seq[1:-1]
    else:
        predicted_text_ids = best_seq[1:]
        
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    
    return predicted_text, hard_pred_color_id, hard_pred_category_id, hard_pred_object_id

In [9]:
# --- Execute Data Setup ---
print("--- 1. Setting up Data and Static Graph ---")
g = torch.Generator().manual_seed(42)
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val   = int(N * VAL_PCT)
n_test  = N - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

# Reduce data for a faster test run, adjust BATCH_SIZE if needed
# n_train_reduced = 500
# n_val_reduced = 50
# n_test_reduced = 50
# train_ds = torch.utils.data.Subset(dataset, range(n_train_reduced))
# val_ds = torch.utils.data.Subset(dataset, range(n_train_reduced, n_train_reduced + n_val_reduced))
# test_ds = torch.utils.data.Subset(dataset, range(n_train_reduced + n_val_reduced, n_train_reduced + n_val_reduced + n_test_reduced))
# n_train = n_train_reduced
# n_val = n_val_reduced
# n_test = n_test_reduced

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
print(f"Data loaders created: Train={n_train}, Val={n_val}, Test={n_test}")

# Create the static Granger Causality Graph
eeg_b, _, _ = next(iter(train_loader))
try:
    granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
    num_channels = eeg_b.shape[1]
    granger_edge_index, granger_edge_attr = add_self_loops(granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels)
    granger_edge_index = granger_edge_index.to(torch.long).to(device)
    granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
    print(f"Static Granger Graph ready on {device}.")
except Exception as e:
    print(f"Could not create Granger Graph (Skipping): {e}")
    # Create dummy graph for non-GCN layers to still run
    num_channels = eeg_b.shape[1]
    granger_edge_index = torch.tensor([[i, i] for i in range(num_channels)], dtype=torch.long).T.to(device)
    granger_edge_attr = torch.ones(num_channels).to(device)

# --- Model and Loss Initialization ---
new_model = Seq2SeqTF(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS, num_categories=NUM_CATEGORIES, num_objects=NUM_OBJECTS,
    d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, pad_id=PAD_ID, dropout=0.1).to(device)

text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.CrossEntropyLoss()
category_criterion = nn.CrossEntropyLoss()
object_criterion = nn.BCEWithLogitsLoss()

# --- Training Loop (50 epochs) ---
print("\n--- 2. Starting Transformer Training (Conditional Training Phase) ---")
new_optimizer = AdamW(new_model.parameters(), lr=3e-5, weight_decay=1e-2)
new_scheduler = ReduceLROnPlateau(new_optimizer, 'min', factor=0.2, patience=2, verbose=True)
EPOCHS = 50
MODEL_SAVE_PATH = 'eeg-meta-text-spatiotemporal-transformer-model-unconditional.pt'
best_val_loss = float('inf')

print(f"Model Parameters: {sum(p.numel() for p in new_model.parameters() if p.requires_grad):,}")

# Continue from previous logs if available (or start fresh)
start_epoch = 1
# if os.path.exists(MODEL_SAVE_PATH):
#     # Load previous state if training was interrupted
#     # ... (code to load state dict and adjust start_epoch)

for epoch in range(start_epoch, EPOCHS + 1):
    start_time = time.time()
    train_loss = train_one_epoch_tf(new_model, train_loader, new_optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    val_loss = evaluate_tf(new_model, val_loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    new_scheduler.step(val_loss)
    end_time = time.time()
    formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
    
    print(f"\nEpoch {epoch:02d}/{EPOCHS} | Time: {formatted_time}")
    print(f"\tTrain Loss: {train_loss:.4f}")
    print(f"\t Val. Loss: {val_loss:.4f} | Val. Perplexity: {math.exp(val_loss):7.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(new_model.state_dict(), MODEL_SAVE_PATH)
        print("\t-> Validation loss improved, saving new best Transformer model. 🏆")
    
    # Check if interrupted based on the trace
    if epoch >= 11:
        # Stop after a few epochs for demonstration/testing purposes
        # If the code was run interactively and interrupted, this block will be skipped
        if 'KeyboardInterrupt' in globals() or 'KeyboardInterrupt' in locals():
             print("Training interrupted. Proceeding to evaluation.")
             break
        
print("\n--- Training Complete/Interrupted. Starting Final Unconditional Evaluation ---")

# --- 3. Final Unconditional Beam Search Evaluation ---
print("\n--- Final UNCONDITIONAL Beam Search & BLEU Evaluation ---")

# Load the best weights
try:
    new_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print(f"Successfully loaded best model weights from: {MODEL_SAVE_PATH}")
except Exception as e:
    print(f"Error loading model weights: {e}. Cannot guarantee final results.")

predictions_list = []
references_list = []
samples_printed = 0
NUM_SAMPLES_TO_PRINT = 10 
global_sample_index = 0
bleu_metric = evaluate.load('bleu')
test_progress_bar = tqdm(test_loader, desc="Unconditional Beam Search (Beam 5)", leave=True)
start_time = time.time()

for eeg_b, meta_b, txt_b in test_progress_bar:
    for i in range(eeg_b.shape[0]):
        eeg_sample = eeg_b[i]
        meta_sample = meta_b[i] # Only used for ground truth metadata comparison
        true_text_ids = txt_b[i]
        
        # --- Generation using UNCONDITIONAL BEAM SEARCH (EEG only) ---
        predicted_text, pred_color_id, pred_category_id, pred_object_id = generate_text_beam_unconditional(
            new_model, eeg_sample, granger_edge_index, granger_edge_attr, beam_width=5
        )
        
        # --- Ground Truth Text and Metadata ---
        true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        true_color_id = int(meta_sample[0].item())
        true_category_id = int(meta_sample[1].item())
        
        if not true_text: continue
            
        # Store for Corpus-Level Evaluation
        predictions_list.append(predicted_text)
        references_list.append([true_text])

        # --- Print Sample Analysis (Qualitative) ---
        if samples_printed < NUM_SAMPLES_TO_PRINT:
            # NOTE: Without your id_to_color/category/object_mapping, we use raw IDs for printing.
            
            true_object_indices = meta_sample[2:].nonzero(as_tuple=True)[0]
            true_object_ids = [idx.item() for idx in true_object_indices]
            if not true_object_ids: true_object_ids = ["None"]
                
            print(f"\n--- Sample {global_sample_index + 1} (Unconditional Beam 5) ---")
            print(f"GROUND TRUTH TEXT: {true_text}")
            print(f"PREDICTED TEXT (EEG-Only): {predicted_text}")
            print("\n  --- METADATA PERFORMANCE (Predicted by CLS Token vs. Truth) ---")
            print(f"  Color:      Truth=ID {true_color_id} | Pred=ID {pred_color_id}")
            print(f"  Category:   Truth=ID {true_category_id} | Pred=ID {pred_category_id}")
            print(f"  Object(s):  Truth={true_object_ids} | Pred=ID {pred_object_id} (Most Likely)")
            samples_printed += 1
            
        global_sample_index += 1

# --- Final Reporting ---
end_time = time.time()
formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list)
final_bleu_score = bleu_results['bleu']

print("\n=============================================")
print(f"Unconditional Evaluation Complete in {formatted_time}")
print("=============================================")
print(f"| Total Samples Evaluated:  {global_sample_index} |")
print("---------------------------------------------")
print(f"| **Corpus-Level BLEU Score (EEG-Only):** {final_bleu_score:.4f} |")
print("=============================================")

--- 1. Setting up Data and Static Graph ---
Data loaders created: Train=22400, Val=2800, Test=2800


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Static Granger Graph ready on cuda.

--- 2. Starting Transformer Training (Conditional Training Phase) ---
Model Parameters: 23,527,913


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]

/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [66,0,0], thread: [96,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [66,0,0], thread: [97,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [66,0,0], thread: [98,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [66,0,0], thread: [99,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [66,0,0], thread: [100,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [66,0,

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
import copy
import time
import math
import statistics
import random
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from torch_geometric.nn import GCNConv
from scipy.sparse import coo_matrix
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import evaluate # For BLEU calculation

# --- Constants & Global Setup ---
# NOTE: Replace these with your actual paths
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_objects_reduced.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased" 

TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Tokenizer and IDs ---
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# --- Metadata Dimensions (Based on your original code) ---
NUM_COLORS = 77
NUM_CATEGORIES = 53
NUM_OBJECTS = 61
D_MODEL = 256 # Transformer hidden dimension

# --- Granger Causality Matrix Creation ---
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j: continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            data = np.vstack([ts_j, ts_i]).T
            if len(data) < 10: continue
                
            try:
                results = grangercausalitytests(data, maxlag=5, verbose=False)
                p_value = results[5][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)
    
    return edge_index.to(torch.long), edge_attr.to(torch.float)

# --- Dataset and DataLoader Components ---
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')
        
        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype('float32'))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype('float32'))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype('int64'))
        
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# --- Transformer Core Components (Utility) ---
def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.linears = get_clones(nn.Linear(d_model, d_model), 4)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        batch_size = query.size(0)
        
        query, key, value = [l(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
                             for l, x in zip(self.linears, (query, key, value))]

        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        p_attn = F.softmax(scores, dim=-1)
        p_attn = self.dropout(p_attn)
        
        x = torch.matmul(p_attn, value)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.linears[-1](x)

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, d_meta):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Metadata Conditional Bias (Semantic Gating)
        self.meta_gate = nn.Sequential(
            nn.Linear(d_meta, d_model),
            nn.Sigmoid()
        )
        
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, src_mask, tgt_mask, meta_features):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, memory, memory, src_mask)))
        
        # Apply Metadata Conditional Bias (Semantic Gating)
        gate = self.meta_gate(meta_features).unsqueeze(1) # (B, 1, D_model)
        x = x * gate 
        
        x = self.norm3(x + self.dropout(self.feed_forward(x)))
        return x

# --- Metadata Encoder ---
class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_categories, num_objects, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.category_embedding = nn.Embedding(num_categories, category_emb_dim)
        
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )
        
        self.output_dim = color_emb_dim + category_emb_dim + object_feature_dim
        self.color_emb_dim = color_emb_dim
        self.category_emb_dim = category_emb_dim
        self.object_feature_dim = object_feature_dim

    def forward(self, metadata):
        color_ids = metadata[:, 0].long()
        category_ids = metadata[:, 1].long()
        object_features_raw = metadata[:, 2:]
        
        object_features_raw = object_features_raw.float()

        color_vec = self.color_embedding(color_ids)
        category_vec = self.category_embedding(category_ids)
        object_vec = self.object_processor(object_features_raw)

        combined_features = torch.cat([color_vec, category_vec, object_vec], dim=1)
        return combined_features

# --- SpatioTemporal EEG Encoder (GCN + Transformer Encoder) ---
class SpatioTemporalEEGEncoderTF(nn.Module):
    def __init__(self, num_channels=62, d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, dropout=0.1):
        super().__init__()
        self.num_channels = num_channels
        self.d_model = d_model
        
        self.gcn1 = GCNConv(num_channels, d_model)
        self.gcn2 = GCNConv(d_model, d_model)
        self.spatial_dropout = nn.Dropout(dropout)

        self.pos_encoding = PositionalEncoding(d_model)

        encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
        self.transformer_layers = get_clones(encoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, eeg, edge_index, edge_attr):
        batch_size, num_channels, num_timesteps = eeg.shape
        
        # --- Spatial Processing (GCN) ---
        batch_edge_index, batch_edge_attr = self._prepare_gcn_input(batch_size, num_timesteps, edge_index, edge_attr)
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels) # (B*T, C)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.spatial_dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))
        
        x = x.reshape(batch_size, num_timesteps, self.d_model)
        
        # --- Transformer Input Preparation ---
        cls_token = self.cls_token.repeat(batch_size, 1, 1) # (B, 1, D_model)
        x = torch.cat([cls_token, x], dim=1) # (B, T+1, D_model)

        x = self.pos_encoding(x)

        # --- Temporal Processing (Transformer Encoder Stack) ---
        for layer in self.transformer_layers:
            x = layer(x)
        
        return self.layer_norm(x)

    # --- REVERTED GCN BATCHING LOGIC (Your Original Working Version) ---
    def _prepare_gcn_input(self, batch_size, num_timesteps, edge_index, edge_attr):        
        # 1. Duplicate graph for BATCH_SIZE
        batch_edge_index = edge_index.repeat(1, batch_size * num_timesteps) 
        batch_edge_attr = edge_attr.repeat(batch_size * num_timesteps)
        
        # 2. Apply offset for each graph instance (B * T)
        batch_offset = torch.arange(batch_size * num_timesteps, device=edge_index.device) * self.num_channels
        batch_edge_index += batch_offset.repeat_interleave(edge_index.shape[1]).unsqueeze(0)
        return batch_edge_index, batch_edge_attr
    # -------------------------------------------------------------

# --- Transformer Decoder ---
class DecoderTF(nn.Module):
    def __init__(self, vocab_size, emb_dim, d_model, num_layers, num_heads, d_ff, pad_id, dropout, d_meta):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.pos_encoding = PositionalEncoding(emb_dim)
        self.input_projection = nn.Linear(emb_dim, d_model) 
        
        decoder_layer = TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, d_meta)
        self.transformer_layers = get_clones(decoder_layer, num_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, target_text_ids, memory, memory_mask, meta_features):
        tgt_embed = self.embedding(target_text_ids)
        x = self.pos_encoding(tgt_embed)
        x = self.input_projection(x)

        tgt_seq_len = target_text_ids.size(1)
        tgt_mask = torch.triu(torch.ones(tgt_seq_len, tgt_seq_len), diagonal=1).bool().to(x.device)
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)
        
        for layer in self.transformer_layers:
            x = layer(x, memory, memory_mask, tgt_mask, meta_features)
        
        x = self.layer_norm(x)
        return self.fc_out(x)
    
class Seq2SeqTF(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_categories, num_objects, 
                 d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, pad_id=PAD_ID, dropout=0.1, 
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        
        self.encoder = SpatioTemporalEEGEncoderTF(
            d_model=d_model, num_layers=num_layers, num_heads=num_heads, d_ff=d_ff, dropout=dropout
        )
        self.meta_encoder = MetadataEncoder(
            num_colors, num_categories, num_objects, color_emb_dim, category_emb_dim, object_feature_dim
        )
        
        meta_features_dim = self.meta_encoder.output_dim
        
        self.decoder = DecoderTF(
            text_vocab_size, d_model, d_model, num_layers, num_heads, d_ff, pad_id, dropout, meta_features_dim
        )
        
        self.meta_head = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_categories + num_objects)
        )
        self.num_colors = num_colors
        self.num_categories = num_categories
        self.num_objects = num_objects
        self.pad_id = pad_id

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr):
        # 1. EEG Encoding (UNCONDITIONAL)
        eeg_features = self.encoder(eeg, edge_index, edge_attr) # (B, T+1, D_model)
        
        # 2. Metadata Encoding (CONDITIONAL - uses ground truth metadata during training)
        meta_features = self.meta_encoder(metadata) # (B, D_meta)
        
        # 3. Auxiliary Metadata Prediction
        cls_token_feature = eeg_features[:, 0, :] # (B, D_model)
        meta_preds = self.meta_head(cls_token_feature)
        
        # 4. Decoding
        decoder_memory = eeg_features[:, 1:, :] # EEG features WITHOUT CLS token
        memory_mask = None 
        
        text_logits = self.decoder(
            target_text[:, :-1], # Target input (shifted right)
            decoder_memory, 
            memory_mask, 
            meta_features # Ground Truth Meta features (for conditional training)
        )
        
        pred_color = meta_preds[:, :self.num_colors]
        pred_category = meta_preds[:, self.num_colors:self.num_colors + self.num_categories]
        pred_object = meta_preds[:, self.num_colors + self.num_categories:]
        
        return text_logits, pred_color, pred_category, pred_object

# --- Training and Evaluation Functions ---
def train_one_epoch_tf(model, loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training TF", leave=False)
    
    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
        optimizer.zero_grad()
        
        text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr)
        
        text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        color_loss = color_criterion(pred_color, meta_b[:, 0].long())
        category_loss = category_criterion(pred_category, meta_b[:, 1].long())
        object_loss = object_criterion(pred_object, meta_b[:, 2:])
        
        loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        
        progress_bar.set_postfix(text_loss=text_loss.item(), meta_loss=loss.item() - text_loss.item())
        
    return total_loss / len(loader)

def evaluate_tf(model, loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr):
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for eeg_b, meta_b, txt_b in loader:
            eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)
            
            text_logits, pred_color, pred_category, pred_object = model(eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr)
            
            text_loss = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
            color_loss = color_criterion(pred_color, meta_b[:, 0].long())
            category_loss = category_criterion(pred_category, meta_b[:, 1].long())
            object_loss = object_criterion(pred_object, meta_b[:, 2:])
            
            loss = text_loss + 0.1 * (color_loss + category_loss) + 0.2 * object_loss
            total_loss += loss.item()
            
    return total_loss / len(loader)

@torch.no_grad()
def generate_text_beam_unconditional(model, eeg_signal, edge_index, edge_attr, 
                                     beam_width=5, max_len=64):    
    """
    Generates text using Beam Search, conditioned ONLY on the EEG signal.
    It uses the model's prediction of metadata from the CLS token to create 
    the meta_features for the decoder's semantic gating.
    """
    model.eval()
    
    eeg_signal = eeg_signal.unsqueeze(0).to(device)
    
    # 1. Encode EEG (UNCONDITIONAL)
    eeg_features = model.encoder(eeg_signal, edge_index, edge_attr)
    decoder_memory = eeg_features[:, 1:, :] 
    memory_mask = None 
    
    # 2. Predict Metadata from CLS token
    cls_token_feature = eeg_features[:, 0, :]
    meta_preds = model.meta_head(cls_token_feature) # (1, Total Meta Size)

    # 3. Create conditioning vector (M_features) from model's predictions
    
    # 3a. Predicted Color and Category IDs (argmax on logits)
    pred_color_id = meta_preds[:, :model.num_colors].argmax(dim=1)
    pred_category_id = meta_preds[:, model.num_colors:model.num_colors + model.num_categories].argmax(dim=1)
    
    # 3b. Predicted Object Logits (soft output)
    pred_object_logits = meta_preds[:, model.num_colors + model.num_categories:] # (1, NUM_OBJECTS)
    
    # 3c. Get the *learned embeddings* and *processed features* using the MetadataEncoder components
    color_vec = model.meta_encoder.color_embedding(pred_color_id)      
    category_vec = model.meta_encoder.category_embedding(pred_category_id) 
    
    # Use the predicted object *logits* as input to the Object Processor
    object_vec = model.meta_encoder.object_processor(pred_object_logits) 

    # 3d. Combine the predicted feature vectors to form the UNCONDITIONAL meta_features
    meta_features = torch.cat([color_vec, category_vec, object_vec], dim=1) # (1, D_meta)
    
    hard_pred_color_id = pred_color_id.item()
    hard_pred_category_id = pred_category_id.item()
    hard_pred_object_id = pred_object_logits.argmax().item()

    # 4. Initialization: Beams list stores (sequence_ids, cumulative_log_probability)
    initial_seq = [SOS_ID]
    beams = [(initial_seq, 0.0)]
    
    # 5. Beam Search Loop
    for _ in range(max_len):
        new_beams = []
        
        for seq, score in beams:
            if seq[-1] == EOS_ID:
                new_beams.append((seq, score))
                continue
                
            input_ids = torch.tensor([seq], dtype=torch.long, device=device)

            # Decoder is conditioned on the UNCONDITIONAL meta_features
            logits = model.decoder(input_ids, decoder_memory, memory_mask, meta_features) 
            next_token_logits = logits[:, -1, :].squeeze(0)
            
            log_probs = F.log_softmax(next_token_logits, dim=-1)
            top_log_probs, top_ids = torch.topk(log_probs, beam_width)
            
            for k in range(beam_width):
                new_token_id = top_ids[k].item()
                new_log_prob = top_log_probs[k].item()
                
                new_seq = seq + [new_token_id]
                new_score = score + new_log_prob 
                
                new_beams.append((new_seq, new_score))

        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        if beams[0][0][-1] == EOS_ID:
            break
            
    # 6. Final Output Selection
    best_seq = beams[0][0]
    
    if best_seq[-1] == EOS_ID:
        predicted_text_ids = best_seq[1:-1]
    else:
        predicted_text_ids = best_seq[1:]
        
    predicted_text = tokenizer.decode(predicted_text_ids, skip_special_tokens=True)
    
    return predicted_text, hard_pred_color_id, hard_pred_category_id, hard_pred_object_id


# --- Execute Data Setup ---
print("--- 1. Setting up Data and Static Graph ---")
g = torch.Generator().manual_seed(42)
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val   = int(N * VAL_PCT)
n_test  = N - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
print(f"Data loaders created: Train={n_train}, Val={n_val}, Test={n_test}")

# Create the static Granger Causality Graph
eeg_b, _, _ = next(iter(train_loader))
try:
    granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
    num_channels = eeg_b.shape[1]
    granger_edge_index, granger_edge_attr = add_self_loops(granger_edge_index, edge_attr=granger_edge_attr, num_nodes=num_channels)
    granger_edge_index = granger_edge_index.to(torch.long).to(device)
    granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
    print(f"Static Granger Graph ready on {device}.")
except Exception as e:
    print(f"Could not create Granger Graph (Skipping): {e}")
    num_channels = eeg_b.shape[1]
    granger_edge_index = torch.tensor([[i, i] for i in range(num_channels)], dtype=torch.long).T.to(device)
    granger_edge_attr = torch.ones(num_channels).to(device)

# --- Model and Loss Initialization ---
new_model = Seq2SeqTF(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS, num_categories=NUM_CATEGORIES, num_objects=NUM_OBJECTS,
    d_model=D_MODEL, num_layers=4, num_heads=8, d_ff=1024, pad_id=PAD_ID, dropout=0.1).to(device)

text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.CrossEntropyLoss()
category_criterion = nn.CrossEntropyLoss()
object_criterion = nn.BCEWithLogitsLoss()

# --- Training Loop (50 epochs) ---
print("\n--- 2. Starting Transformer Training (Conditional Training Phase) ---")
new_optimizer = AdamW(new_model.parameters(), lr=3e-5, weight_decay=1e-2)
new_scheduler = ReduceLROnPlateau(new_optimizer, 'min', factor=0.2, patience=2, verbose=True)
EPOCHS = 50
MODEL_SAVE_PATH = 'eeg-meta-text-spatiotemporal-transformer-model-unconditional.pt'
best_val_loss = float('inf')

print(f"Model Parameters: {sum(p.numel() for p in new_model.parameters() if p.requires_grad):,}")

start_epoch = 1

for epoch in range(start_epoch, EPOCHS + 1):
    start_time = time.time()
    train_loss = train_one_epoch_tf(new_model, train_loader, new_optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    val_loss = evaluate_tf(new_model, val_loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr)
    new_scheduler.step(val_loss)
    end_time = time.time()
    formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
    
    print(f"\nEpoch {epoch:02d}/{EPOCHS} | Time: {formatted_time}")
    print(f"\tTrain Loss: {train_loss:.4f}")
    print(f"\t Val. Loss: {val_loss:.4f} | Val. Perplexity: {math.exp(val_loss):7.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(new_model.state_dict(), MODEL_SAVE_PATH)
        print("\t-> Validation loss improved, saving new best Transformer model. 🏆")
    
    if epoch >= 11: # Placeholder to avoid excessively long interactive runs
         if 'KeyboardInterrupt' in globals() or 'KeyboardInterrupt' in locals():
              print("Training interrupted. Proceeding to evaluation.")
              break
        
print("\n--- Training Complete/Interrupted. Starting Final Unconditional Evaluation ---")

# --- 3. Final Unconditional Beam Search Evaluation ---
print("\n--- Final UNCONDITIONAL Beam Search & BLEU Evaluation ---")

# Load the best weights
try:
    new_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print(f"Successfully loaded best model weights from: {MODEL_SAVE_PATH}")
except Exception as e:
    print(f"Error loading model weights: {e}. Cannot guarantee final results.")

predictions_list = []
references_list = []
samples_printed = 0
NUM_SAMPLES_TO_PRINT = 10 
global_sample_index = 0
bleu_metric = evaluate.load('bleu')
test_progress_bar = tqdm(test_loader, desc="Unconditional Beam Search (Beam 5)", leave=True)
start_time = time.time()

for eeg_b, meta_b, txt_b in test_progress_bar:
    for i in range(eeg_b.shape[0]):
        eeg_sample = eeg_b[i]
        meta_sample = meta_b[i] # Only used for ground truth metadata comparison
        true_text_ids = txt_b[i]
        
        # --- Generation using UNCONDITIONAL BEAM SEARCH (EEG only) ---
        predicted_text, pred_color_id, pred_category_id, pred_object_id = generate_text_beam_unconditional(
            new_model, eeg_sample, granger_edge_index, granger_edge_attr, beam_width=5
        )
        
        # --- Ground Truth Text and Metadata ---
        true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)
        true_color_id = int(meta_sample[0].item())
        true_category_id = int(meta_sample[1].item())
        
        if not true_text: continue
            
        # Store for Corpus-Level Evaluation
        predictions_list.append(predicted_text)
        references_list.append([true_text])

        # --- Print Sample Analysis (Qualitative) ---
        if samples_printed < NUM_SAMPLES_TO_PRINT:
            
            true_object_indices = meta_sample[2:].nonzero(as_tuple=True)[0]
            true_object_ids = [idx.item() for idx in true_object_indices]
            if not true_object_ids: true_object_ids = ["None"]
                
            print(f"\n--- Sample {global_sample_index + 1} (Unconditional Beam 5) ---")
            print(f"GROUND TRUTH TEXT: {true_text}")
            print(f"PREDICTED TEXT (EEG-Only): {predicted_text}")
            print("\n  --- METADATA PERFORMANCE (Predicted by CLS Token vs. Truth) ---")
            print(f"  Color:      Truth=ID {true_color_id} | Pred=ID {pred_color_id}")
            print(f"  Category:   Truth=ID {true_category_id} | Pred=ID {pred_category_id}")
            print(f"  Object(s):  Truth={true_object_ids} | Pred=ID {pred_object_id} (Most Likely)")
            samples_printed += 1
            
        global_sample_index += 1

# --- Final Reporting ---
end_time = time.time()
formatted_time = f"{int((end_time - start_time) // 60):02d}m {int((end_time - start_time) % 60):02d}s"
bleu_results = bleu_metric.compute(predictions=predictions_list, references=references_list)
final_bleu_score = bleu_results['bleu']

print("\n=============================================")
print(f"Unconditional Evaluation Complete in {formatted_time}")
print("=============================================")
print(f"| Total Samples Evaluated:  {global_sample_index} |")
print("---------------------------------------------")
print(f"| **Corpus-Level BLEU Score (EEG-Only):** {final_bleu_score:.4f} |")
print("=============================================")

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


--- 1. Setting up Data and Static Graph ---
Data loaders created: Train=22400, Val=2800, Test=2800


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Static Granger Graph ready on cuda.

--- 2. Starting Transformer Training (Conditional Training Phase) ---
Model Parameters: 23,527,913


/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Training TF:   0%|          | 0/700 [00:00<?, ?it/s]

/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [48,0,0], thread: [96,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [48,0,0], thread: [97,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [48,0,0], thread: [98,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [48,0,0], thread: [99,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [48,0,0], thread: [100,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:94: operator(): block: [48,0,

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
